# S25 · A real A/B test, analysed end to end

A shopping app tests two checkout screens. We generate the data, compute each
conversion rate, run the right test, report the p-value AND a plausible range for the
lift, then make a decision a product manager could actually act on.

**New here? Read this once.**

- New to Python? Press play on each cell, top to bottom, and read the note above each
  one. You will still complete the whole analysis.
- Fuzzy on what a p-value means? Open `primers/p_values_in_plain_words.md` first.
- Already confident? Look for the cell marked **Stretch (optional)** near the end,
  where we show the multiple-testing trap.
- Any unfamiliar word is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib, scipy and statsmodels.
# Google Colab already ships all four, so there is nothing to install.
print("Setup complete - nothing to install.")

## The scenario

A shopping app tests a new checkout screen.

- **Variant A** (control): the current screen, with the usual UPI prompt.
- **Variant B** (new): a redesigned one-tap UPI prompt.

Each user is sent to A or B at random, so the two groups are otherwise alike. Our
metric is **conversion**: did the user finish paying (yes or no)? We want to know if B
genuinely lifts conversion, or if any gap is just luck.

## Step 1 - import our tools

`scipy.stats` for the test, and two `statsmodels` helpers: one runs the test for two
proportions, the other gives a plausible range for the difference between them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# A statsmodels helper that compares two proportions (a two-proportion z-test).
from statsmodels.stats.proportion import proportions_ztest
# A statsmodels helper that gives a confidence interval for the difference.
from statsmodels.stats.proportion import confint_proportions_2indep

# Set a seed so we all get the same random numbers, and the same answers, every time.
np.random.seed(0)

print("Tools ready.")

## Decide the question BEFORE you look

Before generating anything, notice the discipline that makes an A/B test trustworthy:
you fix **one** main metric, and the sample size, *before* the experiment runs. Our
main metric is conversion, full stop. The classic way teams fool themselves is to run
the experiment first, then hunt through a dozen metrics for one that looks good. That
turns luck into a "win". We pick conversion now and stick to it.

## Step 2 - generate the experiment data

We simulate each user as a coin flip: 1 = converted, 0 = did not. Variant A has a true
conversion chance of 11%; Variant B a true chance of 12.8%. In real life we would NOT
know these true values; we set them here only so the lesson is concrete. We use a
large sample so the real lift shows through clearly.

In [ ]:
# How many users each variant received.
users_A = 8000
users_B = 8000

# The TRUE conversion chance for each variant (unknown in real life).
true_rate_A = 0.110
true_rate_B = 0.128

# Simulate each user: 1 if they converted, 0 if not.
# np.random.binomial(1, p, size=n) flips n coins, each with success chance p.
outcomes_A = np.random.binomial(1, true_rate_A, size=users_A)
outcomes_B = np.random.binomial(1, true_rate_B, size=users_B)

# The number who converted is just the sum of the 1s.
converted_A = outcomes_A.sum()
converted_B = outcomes_B.sum()

print("Variant A:", converted_A, "conversions out of", users_A, "users")
print("Variant B:", converted_B, "conversions out of", users_B, "users")

## Step 3 - confirm the split looks balanced

A quick sanity check: the two groups should be roughly the same size. If they were
wildly different, something went wrong with the experiment setup before we even look
at conversion.

In [ ]:
print("users in A:", users_A)
print("users in B:", users_B)
print("the two group sizes are close, so the random split looks fine.")

## Step 4 - compute the conversion rates and the lift

The conversion rate is conversions divided by users. The **lift** is how much higher B
is than A, measured in percentage points.

In [ ]:
rate_A = converted_A / users_A
rate_B = converted_B / users_B

# The observed lift, in percentage points (multiply by 100 to read as %).
lift_points = (rate_B - rate_A) * 100

print("Variant A conversion rate:", round(rate_A * 100, 2), "%")
print("Variant B conversion rate:", round(rate_B * 100, 2), "%")
print("observed lift            :", round(lift_points, 2), "percentage points")

## Step 5 - picture the two conversion rates

A simple bar chart so the gap is easy to see. We write the exact rate on top of each
bar.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(["Variant A", "Variant B"],
        [rate_A * 100, rate_B * 100],
        color=["#2E75B6", "#C0392B"])
plt.ylabel("conversion rate (%)")
plt.title("Conversion rate: A vs B")

# Write the exact rate on top of each bar.
plt.text(0, rate_A * 100 + 0.1, str(round(rate_A * 100, 2)) + "%", ha="center")
plt.text(1, rate_B * 100 + 0.1, str(round(rate_B * 100, 2)) + "%", ha="center")
plt.show()

## Step 6 - run the right test

Our metric is a yes/no event (converted or not), so we compare two **proportions**. We
use a two-proportion z-test, which for this case answers the same question as the
chi-square test from notebook 1.

In [ ]:
# proportions_ztest wants, for each group, the number of successes and the
# total number of trials, each given as an array.
successes = np.array([converted_A, converted_B])
totals = np.array([users_A, users_B])

z_statistic, p_value = proportions_ztest(successes, totals)

print("z-statistic:", round(z_statistic, 3))
print("p-value    :", round(p_value, 4))

# Read the p-value in plain words.
if p_value < 0.05:
    print()
    print("p is below 0.05: the gap is unlikely to be pure luck.")
    print("We have evidence that B really does convert differently from A.")
else:
    print()
    print("p is 0.05 or above: the gap could easily be luck.")
    print("We do NOT have strong evidence of a real difference.")

## Step 7 - a confidence interval for the lift

A p-value gives only a yes/no verdict; it hides HOW BIG the effect is. A **confidence
interval** reports the lift AND its uncertainty: a plausible range for the true
difference. This is what you should actually report to the business.

In [ ]:
# This returns the lower and upper ends of a 95% confidence interval for the
# difference in conversion rates (B minus A).
low, high = confint_proportions_2indep(
    converted_B, users_B,     # B successes and total
    converted_A, users_A,     # A successes and total
    method="wald")

# Convert from a fraction to percentage points so it reads like the lift above.
low_points = low * 100
high_points = high * 100

print("95% confidence interval for the lift (B - A):")
print("from", round(low_points, 2), "to", round(high_points, 2), "percentage points")

## Step 8 - read the interval

If the whole interval sits above 0, we are fairly sure B is better. If the interval
includes 0, we cannot rule out 'no difference'. This always agrees with the p-value at
the same 5% level.

In [ ]:
if low_points > 0:
    print("The whole interval is above 0, so we are fairly confident B is better.")
elif high_points < 0:
    print("The whole interval is below 0, so B looks WORSE than A.")
else:
    print("The interval includes 0, so we cannot be sure B differs from A.")

## Step 9 - unlikely-to-be-luck vs big-enough-to-matter

A result can be **statistically** significant (unlikely to be luck) yet too small to
be **practically** worth shipping. Suppose the business rule is: only ship B if the
lift is at least 1 percentage point. We check the estimated lift against that bar.

In [ ]:
# The smallest lift the business considers worth the engineering effort.
practical_threshold_points = 1.0

print("observed lift           :", round(lift_points, 2), "percentage points")
print("business cares about >=  :", practical_threshold_points, "percentage points")
print()
if lift_points >= practical_threshold_points and low_points > 0:
    print("DECISION: ship B. The lift is statistically real AND big enough to matter.")
elif low_points > 0:
    print("DECISION: B is real but small. Check if it clears the business bar before shipping.")
else:
    print("DECISION: keep A. We do not have evidence that B is genuinely better.")

### Stretch (optional) - the multiple-testing trap

Skip this if you are new to statistics. Here is why we insisted on picking one metric
up front. Imagine the change truly does nothing, and a team eagerly checks twenty
different metrics, each at the 0.05 line. Each single test has a 5% chance of a false
alarm, so across twenty tests you would *expect* about one to look "significant" by
pure luck. Let us watch it happen.

In [ ]:
# Twenty metrics, and B truly equals A on every one of them (no real effect).
number_of_metrics = 20
false_alarms = 0

for metric_number in range(number_of_metrics):
    # Same true conversion chance for both groups: the change does nothing here.
    fake_A = np.random.binomial(1, 0.11, size=4000)
    fake_B = np.random.binomial(1, 0.11, size=4000)

    successes = np.array([fake_A.sum(), fake_B.sum()])
    totals = np.array([len(fake_A), len(fake_B)])
    z_statistic, p_value = proportions_ztest(successes, totals)

    # Count how many "significant" results we get even though nothing is real.
    if p_value < 0.05:
        false_alarms = false_alarms + 1

print("Metrics tested        :", number_of_metrics)
print("'Significant' by luck  :", false_alarms)
print()
print("Nothing was really different, yet some tests still fired.")
print("That is why you fix ONE metric before the experiment, and correct for extras.")

## What you just did

You analysed a full A/B test end to end:

1. fixed one metric before looking,
2. generated realistic conversion data,
3. computed each conversion rate and the lift,
4. chose the right test (two proportions) and read the p-value,
5. reported a **confidence interval** for the lift, and
6. separated **statistical** significance from **practical** importance before
   deciding.

The honest one-line report a manager wants is the decision plus the effect size and
its range, for example: *'Ship B: it lifts conversion by about 1.4 points, 95% CI 0.3
to 2.5.'*

Next notebook: `03_how_many_users_do_we_need.ipynb`, where we work out that sample size
*before* launching, so a test is never too small to see the effect we care about.